In [1]:
#install needed packages
import importlib
import subprocess
import sys

def is_module_available(module_name: str) -> bool:
    """Return True if the importable module exists; handles dotted names safely."""
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

def install_if_missing(pip_name: str, import_name: str = None):
    """
    Install `pip_name` only if `import_name` (or `pip_name` if None) is not importable.
    Use this to handle cases where pip/distribution names differ from import names.
    """
    import_name = import_name or pip_name
    if not is_module_available(import_name):
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
    else:
        print(f"{pip_name} already installed.")

# --- Packages ---
# simple 1:1 cases
install_if_missing("keyring", "keyring")
install_if_missing("ipython-secrets", "ipython_secrets")
install_if_missing("starrocks", "starrocks")
install_if_missing("oauth2client", "oauth2client")
install_if_missing("sqlalchemy", "sqlalchemy")
install_if_missing("pymysql", "pymysql")
install_if_missing("pyarrow", "pyarrow")
install_if_missing("fsspec", "fsspec")
install_if_missing("s3fs", "s3fs")
install_if_missing("statsmodels", "statsmodels")
install_if_missing("matplotlib", "matplotlib")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("xgboost", "xgboost")

# special cases
# keyrings.cryptfile installs a plugin under the 'keyrings' package; checking the root avoids ModuleNotFoundError
install_if_missing("keyrings.cryptfile", "keyrings")

# Core viz + scaling pins for Python 3.11.11
install_if_missing("bokeh==3.6.2", "bokeh")
install_if_missing("jupyter_bokeh", "jupyter_bokeh")
install_if_missing("panel==1.5.2", "panel")
install_if_missing("holoviews==1.19.0", "holoviews")
install_if_missing("hvplot==0.10.0", "hvplot")
install_if_missing("datashader==0.16.3", "datashader")
install_if_missing("dask[dataframe]==2024.9.1", "dask")
install_if_missing("distributed==2024.9.1", "distributed")
install_if_missing("reportlab", "reportlab")

print("\n✓ All packages installed/verified")

keyring already installed.
ipython-secrets already installed.
starrocks already installed.
oauth2client already installed.
sqlalchemy already installed.
pymysql already installed.
pyarrow already installed.
fsspec already installed.
s3fs already installed.
statsmodels already installed.
matplotlib already installed.
scikit-learn already installed.
xgboost already installed.
keyrings.cryptfile already installed.
bokeh==3.6.2 already installed.
jupyter_bokeh already installed.
panel==1.5.2 already installed.
holoviews==1.19.0 already installed.
hvplot==0.10.0 already installed.
datashader==0.16.3 already installed.
dask[dataframe]==2024.9.1 already installed.
distributed==2024.9.1 already installed.
reportlab already installed.

✓ All packages installed/verified


In [2]:
import keyring
import os
from getpass import getpass
from keyrings.cryptfile.cryptfile import CryptFileKeyring
from pathlib import Path

# Use home directory dynamically instead of hardcoded path
home_dir = str(Path.home())
os.environ["KEYRING_CRYPTFILE_PATH"] = f"{home_dir}/.local/share/python_keyring/cryptfile_pass.cfg"

kr = CryptFileKeyring()
kr.keyring_key = getpass("Set/enter master password for encrypted keyring: ")
keyring.set_keyring(kr)

# Prompt for credentials if not already stored
starrocks_username = keyring.get_password("starrocks", "username")
starrocks_password = keyring.get_password("starrocks", "password")

if not starrocks_username:
    starrocks_username = input("Enter StarRocks username: ")
    keyring.set_password("starrocks", "username", starrocks_username)

if not starrocks_password:
    starrocks_password = getpass("Enter StarRocks password: ")
    keyring.set_password("starrocks", "password", starrocks_password)

In [3]:
import pandas as pd
from sqlalchemy import create_engine, text
from ipython_secrets import get_secret

starrocks_username = keyring.get_password("starrocks", "username")
starrocks_password = keyring.get_password("starrocks", "password")
assert starrocks_username and starrocks_password, "No creds in keyring."

host = "kube-starrocks-warehouse-fe-service.starrocks.svc.cluster.local"
port = 9030
database = "sandbox_finance"

# NOTE: use mysql+pymysql instead of starrocks://
engine = create_engine(
    f"mysql+pymysql://{starrocks_username}:{starrocks_password}@{host}:{port}/{database}",
    connect_args={
        "connect_timeout": 5,
        "read_timeout": 600,
        "write_timeout": 60,
    },
    pool_pre_ping=True,
)

# sanity check
with engine.connect() as conn:
    conn.execute(text("SET query_timeout = 5"))
    conn.execute(text("SELECT 1"))


In [4]:
#pull in data with CLUSTER resources using Enterprise Gateway (SparkCaster)
# Enterprise Gateway endpoint: spark-gateway.kubeflow.svc.cluster.local:8888
from spark.starrocks import StarRocksSparkClient
from pyspark.sql import SparkSession

# Configure Spark to use Enterprise Gateway (SparkCaster) on port 8888
spark = SparkSession.builder \
    .appName("StarRocksApp") \
    .config("spark.gateway.host", "spark-gateway.kubeflow.svc.cluster.local") \
    .config("spark.gateway.port", "8888") \
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.instances", "20") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "10000") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "5") \
    .config("spark.dynamicAllocation.maxExecutors", "25") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print("="*60)
print("SPARK CLUSTER CONFIGURATION - ENTERPRISE GATEWAY")
print("="*60)
print(f"Gateway Host: spark-gateway.kubeflow.svc.cluster.local:8888")
print(f"Executor Memory: {spark.conf.get('spark.executor.memory')}")
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Executor Cores: {spark.conf.get('spark.executor.cores')}")
print(f"Executor Instances: {spark.conf.get('spark.executor.instances')}")
print(f"Dynamic Allocation: {spark.conf.get('spark.dynamicAllocation.enabled')}")
print(f"Min/Max Executors: {spark.conf.get('spark.dynamicAllocation.minExecutors')}/{spark.conf.get('spark.dynamicAllocation.maxExecutors')}")
print("="*60)
    
#set up starrocks spark client with memory optimization options
star = StarRocksSparkClient(
    starrocks_user=starrocks_username,
    starrocks_password=starrocks_password,
    df_reader_options={
        "starrocks.exec.mem.limit": "107374182400",      # 100GB per query
        "starrocks.request.tablet.size": "2",          # more parallelism, smaller chunks  
        "starrocks.batch.size": "2048",                # smaller batches (default 4096)
        "starrocks.request.query.timeout.s": "3600",   # 2 hour timeout
    }
)

# turn off warnings
spark.sparkContext.setLogLevel("ERROR")


#df.count()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 04:35:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SPARK CLUSTER CONFIGURATION - ENTERPRISE GATEWAY
Gateway Host: spark-gateway.kubeflow.svc.cluster.local:8888


2026-02-25 04:35:03,144 - INFO - 


Initialized StarRocksSparkClient with user `jbok`





Executor Memory: 16g
Driver Memory: 8g
Executor Cores: 4
Executor Instances: 20
Dynamic Allocation: true
Min/Max Executors: 5/25


In [5]:
#SQL query for DCGM union and dedupe

df=star.sql(
    '''-- add 6 more dcgm metrics(memory, sm, vram), paramatize start and end dates

-- CREATE OR REPLACE VIEW sandbox_finance.v_capacity_finance_dcgm_metrics_union_raw_v3_1 AS

WITH params AS (
  SELECT
    CAST('2025-09-04 00:00:00' AS DATETIME) AS start_ts,
    -- CAST('2025-01-23 00:00:00' AS DATETIME) AS start_ts,
    /* set to NULL now; fill later if you want a cap */
    CAST(NULL AS DATETIME)          AS end_ts
    -- CAST('2025-09-04 00:00:00' AS DATETIME) AS end_ts

),
    dcgm_gpu_util_avg as(
        select
            datestamp,
            cluster,
            cluster_org,
            instance,
     -- bucket early to reduce cardinality
            node,
            pod,
            region,
            zone,
            'gpu_util' as label,
            avg(value/100) as value
        from
            staging_victoria_metrics.dcgm_fi_dev_gpu_util
        where node is not null
            AND datestamp >= (SELECT start_ts FROM params)
            AND ( (SELECT end_ts FROM params) IS NULL
            OR datestamp <  (SELECT end_ts FROM params) )
        group by 1,2,3,4,5,6,7,8,9
    ), dcgm_gpu_util_deduped as (
        select
            datestamp,
            node,
            label,
            max(cluster) as cluster,
            max(cluster_org) as cluster_org,
            max(instance) as instance,

            max(pod) as pod,
            max(region) as region,
            max(zone) as zone,
            max(value) as value
        from dcgm_gpu_util_avg
        group by 1,2,3
),
-- 1) Aggregate each DCGM metric to node-grain with P50/P95 (StarRocks: percentile_approx)
dcgm_gpu_util AS (
  SELECT
    datestamp,
    node,
    region,
    cluster,
    cluster_org,
    label,

    value
  FROM dcgm_gpu_util_deduped

),
    dcgm_tensor_util_avg as(
        select
            datestamp,
            cluster,
            cluster_org,
            instance,
            node,
            pod,
            region,
            zone,
            'tensor_util' as label,
            avg(value) as value
        from
            staging_victoria_metrics.dcgm_fi_prof_pipe_tensor_active
        where node is not null
            AND datestamp >= (SELECT start_ts FROM params)
            AND ( (SELECT end_ts FROM params) IS NULL
            OR datestamp <  (SELECT end_ts FROM params) )
            -- and region not in ('LAS1','ORD1','RDU1')
        group by 1,2,3,4,5,6,7,8,9
    ), dcgm_tensor_util_deduped as (
        select
            datestamp,
            node,
            label,
            max(cluster) as cluster,
            max(cluster_org) as cluster_org,
            max(instance) as instance,

            max(pod) as pod,
            max(region) as region,
            max(zone) as zone,
            max(value) as value
        from dcgm_tensor_util_avg
        group by 1,2,3
),
-- 1) Aggregate each DCGM metric to node-grain with P50/P95 (StarRocks: percentile_approx)
dcgm_tensor_util AS (
  SELECT
    datestamp,
    node,
    region,
    cluster,
    cluster_org,
    label,

    value
  FROM dcgm_tensor_util_deduped

),
    dcgm_chip_power_avg as(
        select
            datestamp,
            cluster,
            cluster_org,
            instance,

            node,
            pod,
            region,
            zone,
            'chip_power' as label,
            sum(value) as value
        from
            staging_victoria_metrics.dcgm_fi_dev_power_usage
        where node is not null
            AND datestamp >= (SELECT start_ts FROM params)
            AND ( (SELECT end_ts FROM params) IS NULL
            OR datestamp <  (SELECT end_ts FROM params) )
            -- and region not in ('LAS1','ORD1','RDU1')
        group by 1,2,3,4,5,6,7,8,9
    ), dcgm_chip_power_deduped as (
        select
            datestamp,
            node,
            label,
            max(cluster) as cluster,
            max(cluster_org) as cluster_org,
            max(instance) as instance,

            max(pod) as pod,
            max(region) as region,
            max(zone) as zone,
            max(value) as value
        from dcgm_chip_power_avg
        group by 1,2,3
),
-- 1) Aggregate each DCGM metric to node-grain with P50/P95 (StarRocks: percentile_approx)
dcgm_chip_power AS (
  SELECT
    datestamp,
    node,
    region,
    cluster,
    cluster_org,
    label,

    value
  FROM dcgm_chip_power_deduped

),
/*
    dcgm_gpu_vram_avg as(
    select
        datestamp,
        cluster,
        cluster_org,
        instance,

        node,
        pod,
        region,
        zone,
        'vram_usage' as label,
        sum(value) as value
    from
        dcgm_fi_dev_fb_used
    where node is not null
        AND datestamp >= (SELECT start_ts FROM params)
        AND ( (SELECT end_ts FROM params) IS NULL
        OR datestamp <  (SELECT end_ts FROM params) )
    group by 1,2,3,4,5,6,7,8,9
    ), dcgm_gpu_vram_deduped as (
        select
            datestamp,
            node,
            label,
            max(cluster) as cluster,
            max(cluster_org) as cluster_org,
            max(instance) as instance,

            max(pod) as pod,
            max(region) as region,
            max(zone) as zone,
            max(value) as value
        from dcgm_gpu_vram_avg
        group by 1,2,3
),
    dcgm_gpu_vram_usage AS (
        SELECT datestamp,
                 node,
                 region,
                 cluster,
                 cluster_org,
                 label,

                 value
        FROM dcgm_gpu_vram_deduped
),
    dcgm_mem_copy_util_avg as(
    select
        datestamp,
        cluster,
        cluster_org,
        instance,

        node,
        pod,
        region,
        zone,
        'mem_copy_util' as label,
        avg(value/100) as value
    from
        dcgm_fi_dev_mem_copy_util
    where node is not null
        AND datestamp >= (SELECT start_ts FROM params)
        AND ( (SELECT end_ts FROM params) IS NULL
        OR datestamp <  (SELECT end_ts FROM params) )
    group by 1,2,3,4,5,6,7,8,9
),
dcgm_mem_copy_util_deduped as (
    select
        datestamp,
        node,
        label,
        max(cluster) as cluster,
        max(cluster_org) as cluster_org,
        max(instance) as instance,

        max(pod) as pod,
        max(region) as region,
        max(zone) as zone,
        max(value) as value
    from dcgm_mem_copy_util_avg
    group by 1,2,3
),
    dcgm_mem_copy_util AS (
        SELECT datestamp,
                 node,
                 region,
                 cluster,
                 cluster_org,
                 label,

                 value
        FROM dcgm_mem_copy_util_deduped
),
dcgm_gpu_sm_clock_avg as(
select
    datestamp,
    cluster,
    cluster_org,
    instance,

    node,
    pod,
    region,
    zone,
    'sm_clock' as label,
    avg(value) as value
from
    dcgm_fi_dev_sm_clock
where node is not null
    AND datestamp >= (SELECT start_ts FROM params)
    AND ( (SELECT end_ts FROM params) IS NULL
    OR datestamp <  (SELECT end_ts FROM params) )
group by 1,2,3,4,5,6,7,8,9
),
    dcgm_gpu_sm_clock_deduped as (
        select
            datestamp,
            node,
            label,
            max(cluster) as cluster,
            max(cluster_org) as cluster_org,
            max(instance) as instance,

            max(pod) as pod,
            max(region) as region,
            max(zone) as zone,
            max(value) as value
        from dcgm_gpu_sm_clock_avg
        group by 1,2,3
),
dcgm_gpu_sm_clock AS (
    SELECT datestamp,
             node,
             region,
             cluster,
             cluster_org,
             label,

             value
    FROM dcgm_gpu_sm_clock_deduped
    ),
    dcgm_dram_active_avg as(
    select
        datestamp,
        cluster,
        cluster_org,
        instance,

        node,
        pod,
        region,
        zone,
        'dram_active' as label,
        avg(value) as value
    from
        dcgm_fi_prof_dram_active
    where node is not null
        AND datestamp >= (SELECT start_ts FROM params)
        AND ( (SELECT end_ts FROM params) IS NULL
        OR datestamp <  (SELECT end_ts FROM params) )
    group by 1,2,3,4,5,6,7,8,9
    ),
    dcgm_dram_active_deduped as (
        select
            datestamp,
            node,
            label,
            max(cluster) as cluster,
            max(cluster_org) as cluster_org,
            max(instance) as instance,

            max(pod) as pod,
            max(region) as region,
            max(zone) as zone,
            max(value) as value
        from dcgm_dram_active_avg
        group by 1,2,3
),
    dcgm_dram_active AS (
        SELECT datestamp,
                 node,
                 region,
                 cluster,
                 cluster_org,
                 label,

                 value
        FROM dcgm_dram_active_deduped
    ),
    dcgm_gpu_sm_active_avg as(
    select
        datestamp,
        cluster,
        cluster_org,
        instance,

        node,
        pod,
        region,
        zone,
        'sm_active' as label,
        avg(value) as value
    from
        dcgm_fi_prof_sm_active
    where node is not null
        AND datestamp >= (SELECT start_ts FROM params)
        AND ( (SELECT end_ts FROM params) IS NULL
        OR datestamp <  (SELECT end_ts FROM params) )
    group by 1,2,3,4,5,6,7,8,9
    ),
    dcgm_gpu_sm_active_deduped as (
        select
            datestamp,
            node,
            label,
            max(cluster) as cluster,
            max(cluster_org) as cluster_org,
            max(instance) as instance,

            max(pod) as pod,
            max(region) as region,
            max(zone) as zone,
            max(value) as value
        from dcgm_gpu_sm_active_avg
        group by 1,2,3
    ),
    dcgm_gpu_sm_active AS (
        SELECT datestamp,
                 node,
                 region,
                 cluster,
                 cluster_org,
                 label,

                 value
        FROM dcgm_gpu_sm_active_deduped
),
    dcgm_gpu_sm_occupancy_avg as(
    select
        datestamp,
        cluster,
        cluster_org,
        instance,

        node,
        pod,
        region,
        zone,
        'sm_occupancy' as label,
        avg(value) as value
        from
            dcgm_fi_prof_sm_occupancy
        where node is not null
            AND datestamp >= (SELECT start_ts FROM params)
            AND ( (SELECT end_ts FROM params) IS NULL
            OR datestamp <  (SELECT end_ts FROM params) )
        group by 1,2,3,4,5,6,7,8,9
    ),
    dcgm_gpu_sm_occupancy_deduped as (
        select
            datestamp,
            node,
            label,
            max(cluster) as cluster,
            max(cluster_org) as cluster_org,
            max(instance) as instance,

            max(pod) as pod,
            max(region) as region,
            max(zone) as zone,
            max(value) as value
        from dcgm_gpu_sm_occupancy_avg
        group by 1,2,3
),
    dcgm_gpu_sm_occupancy AS (
        SELECT datestamp,
                 node,
                 region,
                 cluster,
                 cluster_org,
                 label,

                 value
        FROM dcgm_gpu_sm_occupancy_deduped
        )
        */
    dcgm_union_all as (
        select datestamp, node, region, cluster, cluster_org, label, value from dcgm_gpu_util
        union all
        select datestamp, node, region, cluster, cluster_org , label, value from dcgm_tensor_util
        union all
        select datestamp, node, region, cluster, cluster_org,  label, value from dcgm_chip_power
        /*
        union all
        select datestamp, node, region, cluster, cluster_org,  label, value from dcgm_gpu_sm_clock
        union all
        select datestamp, node, region, cluster, cluster_org,  label, value from dcgm_dram_active
        union all
        select datestamp, node, region, cluster, cluster_org,  label, value from dcgm_gpu_sm_active
        union all
        select datestamp, node, region, cluster, cluster_org,  label, value from dcgm_gpu_sm_occupancy
        union all
        select datestamp, node, region, cluster, cluster_org,  label, value from dcgm_gpu_vram_usage
        union all
        select datestamp, node, region, cluster, cluster_org,  label, value from dcgm_mem_copy_util
        */
    ),

-- 2) Build the key set (all datestamp,node that appear in any metric)
dcgm_keys AS (
  SELECT DISTINCT datestamp, node, region, cluster, cluster_org, label
  FROM dcgm_union_all
),

-- 3) Assemble the wide DCGM row
dcgm_wide AS (
  SELECT
    k.datestamp,
    k.node,
    k.region,
    k.cluster,
    k.cluster_org,

    k.label,
    u.value
    -- u.gpu_util_p50,
    -- u.gpu_util_p95
  FROM dcgm_keys k
  LEFT JOIN dcgm_union_all u
    ON  k.datestamp   = u.datestamp
    AND k.node        = u.node
    AND k.region      = u.region
    AND k.cluster     = u.cluster
    AND k.cluster_org = u.cluster_org

    AND k.label       = u.label
),
dcgm_pivot AS (
  SELECT
    datestamp, node, region, cluster, cluster_org,
    MAX(CASE WHEN label = 'gpu_util'    THEN value END) AS gpu_util,
    MAX(CASE WHEN label = 'tensor_util' THEN value END) AS tensor_util,
    MAX(CASE WHEN label = 'chip_power'  THEN value END) AS chip_power
    /*
    ,
    MAX(CASE WHEN label = 'vram_usage'  THEN value END) AS vram_usage,
    MAX(CASE WHEN label = 'mem_copy_util'  THEN value END) AS mem_copy_util,
    MAX(CASE WHEN label = 'dram_active'  THEN value END) AS dram_active,
    MAX(CASE WHEN label = 'sm_clock'  THEN value END) AS sm_clock,
    MAX(CASE WHEN label = 'sm_active'  THEN value END) AS sm_active,
    MAX(CASE WHEN label = 'sm_occupancy'  THEN value END) AS sm_occupancy
    */

  FROM dcgm_wide
  GROUP BY datestamp, node, region, cluster, cluster_org
),
bmn_candidates AS (
  SELECT datestamp, node, cluster, model, COUNT(*) AS slot_count
  FROM staging_victoria_metrics.baremetal_node_info
  WHERE 1=1
    -- and datestamp > '2025-01-23 00:00:00'
    AND deviceslot IS NOT NULL
  GROUP BY datestamp, node, cluster, model
),
bmn_winners AS (
  SELECT datestamp, node, cluster, MAX(slot_count) AS max_slots
  FROM bmn_candidates
  GROUP BY datestamp, node, cluster
),
bmn_resolved AS (
  SELECT
      c.datestamp,
      c.node,
      c.cluster,
      MIN(c.model) AS resolved_model   -- tie-breaker: lexicographic
  FROM bmn_candidates c
  JOIN bmn_winners w
    ON c.datestamp = w.datestamp
   AND c.node      = w.node
   AND c.cluster   = w.cluster
   AND c.slot_count = w.max_slots
  GROUP BY c.datestamp, c.node, c.cluster
),
bmn_all_ts AS (
  SELECT DISTINCT datestamp, node, cluster, serial, cw_sku, model
  FROM staging_victoria_metrics.baremetal_node_info
  WHERE 1=1
    -- AND datestamp > '2025-01-23 00:00:00'
    AND deviceslot IS NOT NULL
),
bmn_joined AS (
  SELECT
      a.datestamp,
      a.node,
      a.cluster,
      a.serial,
      a.cw_sku,
      a.model,
      r.resolved_model
  FROM bmn_all_ts a
  LEFT JOIN bmn_resolved r
    ON a.datestamp = r.datestamp
   AND a.node      = r.node
   AND a.cluster   = r.cluster
),
bmn_collapsed AS (
  SELECT
      datestamp,
      node,
      cluster,
      MAX(serial) AS serial   -- or MAX/ANY_VALUE depending on your rule
  FROM bmn_joined
  GROUP BY datestamp, node, cluster
),
baremetal_cleaned AS (
  SELECT
      j.datestamp,
      j.node,
      j.cluster,
      j.cw_sku,
      j.model,
      c.serial,
      LAST_VALUE(CASE WHEN j.resolved_model IS NOT NULL THEN j.resolved_model END)
        OVER (
          PARTITION BY j.node, j.cluster
          ORDER BY j.datestamp
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS final_model
  FROM bmn_joined j
  JOIN bmn_collapsed c
    ON j.datestamp = c.datestamp
   AND j.node      = c.node
   AND j.cluster   = c.cluster
  WHERE j.node IS NOT NULL
),

-- 5) Redfish aggregated to (datestamp, serial)
redfish_grouped AS (
    -- This CTE gets distinct records from the source table.
    -- The original syntax was valid but verbose.
    SELECT DISTINCT
        datestamp,
        -- super_region,
        region,
        zone,
        -- model,
        cluster,
        cluster_org,
        vendor,
        bmc_ip,
        device_type,
        deviceslot,
        device_model,
        serial,
        value
    FROM
        staging_victoria_metrics.redfish_stats_power_reading
    where
        serial is not null
        and serial !='N/A'
        and serial !='�������'
        and value != 0
),

redfish_deduped AS (
    -- This CTE aggregates the data to get the average 'value' per device ('serial') per day.
    SELECT
        datestamp,
        serial,
        -- MAX(super_region) AS super_region,
        MAX(region) AS region,
        MAX(zone) AS zone,
        -- MAX(model) AS model,
        MAX(cluster) AS cluster,
        MAX(cluster_org) AS cluster_org,
        MAX(vendor) AS vendor,
        MAX(bmc_ip) AS bmc_ip,
        MAX(device_type) AS device_type,
        MAX(deviceslot) AS deviceslot,
        MAX(device_model) AS device_model,
        max(value) AS redfish_power
    FROM
        redfish_grouped
    GROUP BY
        datestamp,
        serial
),
customer as (
    SELECT * FROM model_conformed.dim_customer
),
fe_base AS (
  SELECT
    d.datestamp,
    d.node,
    d.gpu_util,
    d.tensor_util,
    d.chip_power,
    /*
    d.dram_active,
    d.mem_copy_util,
    d.vram_usage,
    d.sm_active,
    d.sm_clock,
    d.sm_occupancy,
    */
    d.region,
    d.cluster,
    d.cluster_org,
    b.cw_sku,
    b.model,
    b.serial,
    r.redfish_power,
    c.customer_name,
    c.flag_is_coreweave,

    COALESCE(
      b.model,
      CASE WHEN d.region='RDU1' THEN 'GPU-H100'
           WHEN d.region='ORD1' THEN 'GPU-A100'
           WHEN d.region='LAS1' THEN 'GPU-H100'
           ELSE NULL END
    ) AS model_imputed,
    CASE
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%vr_nvl144%' THEN 'VR100'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%l40s%'      THEN 'L40S'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%l40%'       THEN 'L40'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%gh200%'     THEN 'GH200'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%h100%'      THEN 'H100'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%gb200%'     THEN 'GB200'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%b200%'      THEN 'B200'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%b40%'       THEN 'B40'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%h200%'      THEN 'H200'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%a100%'      THEN 'A100'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%gb300%'     THEN 'GB300'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%b300%'      THEN 'B300'
      WHEN LOWER(COALESCE(b.model,'')) LIKE '%a40%' OR LOWER(COALESCE(b.model,'')) LIKE '%rtx%'
        OR LOWER(COALESCE(b.model,'')) LIKE '%tesla%' OR LOWER(COALESCE(b.model,'')) LIKE '%v100%'
        OR LOWER(COALESCE(b.model,'')) LIKE '%a5000%' OR LOWER(COALESCE(b.model,'')) LIKE '%a6000%'
        THEN 'Legacy'
      WHEN LOWER(COALESCE(b.model,'')) LIKE 'cpu%'        THEN 'CPU'
      WHEN LOWER(COALESCE(b.model,'')) LIKE 'gpu%'        THEN 'Other GPU'
      ELSE 'Other'
    END AS product
  FROM dcgm_pivot d
  LEFT JOIN baremetal_cleaned b ON d.datestamp=b.datestamp AND d.node=b.node
  LEFT JOIN redfish_deduped r       ON d.datestamp=r.datestamp AND b.serial=r.serial
  LEFT JOIN customer c          on d.cluster_org=c.org_id
),


    -- 3) Final: also compute per-row TFLOPS (tensor utilization × peak)
final_enriched AS (SELECT f.*,
                          CASE
                              WHEN f.product IN ('Other', 'Other GPU', 'Legacy')
                                  AND f.model_imputed = 'GPU-A100' THEN 'A100'
                              WHEN f.product IN ('Other', 'Other GPU', 'Legacy')
                                  AND f.model_imputed = 'GPU-H100' THEN 'H100'

                              ELSE f.product
                              END AS product_resolved,
                          case
                                when f.customer_name in ('Albatross','Condor','Eagle','Osprey','Snipe','Heron','Phoenix','Private','Meta') THEN 'Bigbird'
                                when f.customer_name in ('Mistral AI','Cohere') THEN 'AI Lab'
                                when f.customer_name in ('Weights & Biases') THEN 'Internal'
                                when lower(f.customer_name) like '%coreweave%' then 'Internal'
                                else 'External-Other'
                                End as customer_segment
                   FROM fe_base f),
-- 2) Add peak_tflops_unit (based on product)

product_specs AS (
  SELECT * FROM (
    VALUES
      ('H100',  8, 1979,  1100, 1980,'HGX'),
      ('H200',  8, 1979,  1100, 1980, 'HGX'),
      ('A100',  8,  624,  1100, 1410, 'HGX'),
      ('L40S',  8,  733,  600, 2520, 'PCIE'),
      ('L40',   8,  362,  600, 2490,'PCIE'),
      ('A40',   8,  299,  500, 1740,'PCIE'),
      ('GH200', 8, 1979,  1100, 1980, 'MGX'),
      ('GB200', 4, 5000, 1500, 1965, 'MGX'),
      ('B200',  8, 4500, 1100, 1965, 'HGX'),
      ('B40',   8,  936,  1100, 1900,'PCIE'),
      ('GB300', 4, 5000, 1500, 1965, 'MGX'),
      ('B300',  8, 4500, 1900, 1965, 'HGX'),
      ('R100',  8,9000, 2500, 2000,'HGX'),
      ('VR100', 8,10000, 3500, 2000,'MGX'),
      ('B10',   8,484, 300, 1000,'Workstation'),
      ('R10',   8,629, 300, 1000,'Workstation')
  ) AS t(product_resolved, expected_gpus, peak_tflops_unit, peak_power_unit, peak_sm_clock, product_segment)
),
/*    fe_with_peak AS (
  SELECT
    base.*,
    CASE
      WHEN product_resolved='H100'  THEN 1979
      WHEN product_resolved='H200'  THEN 1979
      WHEN product_resolved='A100'  THEN 624
      WHEN product_resolved='L40S'  THEN 733
      WHEN product_resolved='L40'   THEN 362
      WHEN product_resolved='GH200' THEN 1979
      WHEN product_resolved='GB200' THEN 5000
      WHEN product_resolved='B200'  THEN 4500
      WHEN product_resolved='B40'   THEN 936
      WHEN product_resolved='GB300' THEN 5000
      WHEN product_resolved='B300'  THEN 4500
      WHEN product_resolved='VR100' THEN 10000
      ELSE NULL
    END AS peak_tflops_unit
  FROM final_enriched base
),*/
kube_lbl_0 as (
    select distinct
        a.datestamp,
        a.node,
        a.cluster_org,
        a.region
    from staging_victoria_metrics.kube_pod_info a
    join staging_victoria_metrics.kube_pod_labels b
           using (datestamp, cluster, cluster_org, instance, namespace)
     where 1=1
        and a.namespace = 'tenant-slurm'
        and b.namespace = 'tenant-slurm'
        and node is not null
    -- group by 1,2
    -- limit 100
),
    kube_lbl_1 as (
    select
        datestamp,
        node,
        max(cluster_org) as cluster_org,
        max(region) as region
    from kube_lbl_0
    group by 1,2
),
 infini_1_1 as (
        select datestamp, cluster_org, node, region, sum(value) as val
        from staging_victoria_metrics.node_infiniband_port_data_received_bytes_total
        where node is not null
        group by 1,2, 3,4
),
    infini_1_2 as (
        select
            datestamp,
            node,
            max(region) as region,
            max(cluster_org) as cluster_org,
            sum(val) as val
        from infini_1_1
    group by 1,2
),
  infini_2_1 as (
        select datestamp, cluster_org, node, region, sum(value) as val
        from staging_victoria_metrics.node_infiniband_port_data_transmitted_bytes_total
        where node is not null
        group by 1,2, 3,4
),
    infini_2_2 as (
        select
            datestamp,
            node,
            max(region) as region,
            max(cluster_org) as cluster_org,
            sum(val) as val
        from infini_2_1
    group by 1,2
),
    infini_union as (
        select *
        from infini_1_2
        union all
        select *
        from infini_2_2
),
    infini_consolidated as (
        select
            datestamp,
            node,
            cluster_org,
            region,
            sum(val) as value
        from infini_union
        group by 1,2,3,4
),
    -- (unchanged)
final_enriched_resolved AS (
  SELECT
    f.*,
    CASE
        WHEN bb.node IS NOT NULL THEN 'slurm'
        ELSE NULL
    END AS is_training,
    CASE
        WHEN cc.node IS NOT NULL THEN 'multi'
        ELSE NULL
    END AS is_multinode,
    ps.expected_gpus       AS gpu_count_expected,
    ps.peak_tflops_unit,
    ps.peak_power_unit,
    ps.product_segment,
    /*
    dram_active/nullif(sm_active,0)                                                         as memory_boundness_index,  -- bottle neck detector high > 1.0 memory bound, low <1.0 compute bound
    (ps.peak_tflops_unit * f.tensor_util * ps.expected_gpus)/nullif(chip_power,0)           as TFLOPS_per_watt_efficiency, -- higher number means more efficient compute per watt
    sm_occupancy/nullif(sm_active,0)                                                        as compute_occupancy_index, -- how much SM compute space is filled
    tensor_util/nullif(dram_active,0)                                                       as tensor_util_dram_index, -- high > 1.0 compute bound, low <0.8 memory bbound
    tensor_util * ps.peak_tflops_unit * sm_clock/ps.peak_sm_clock * ps.expected_gpus        as tensor_tflops_sm, -- sm clock speed based tflops
    */
    -- per-row instantaneous TFLOPS (row grain = node×ts)
    (ps.peak_tflops_unit * f.tensor_util * ps.expected_gpus)                                AS tensor_tflops,
    (ps.peak_power_unit * ps.expected_gpus)                                                 AS gpu_peak_power_node_w
  FROM final_enriched f
  LEFT JOIN product_specs ps
    ON f.product_resolved = ps.product_resolved
  LEFT JOIN kube_lbl_1 bb
    ON f.node      = bb.node
   AND f.datestamp = bb.datestamp
    and f.region = bb.region
    and f.cluster_org = bb.cluster_org
  LEFT JOIN infini_consolidated cc
    ON f.node      = cc.node
   AND f.datestamp = cc.datestamp
    and f.region = cc.region
    and f.cluster_org = cc.cluster_org
)

select * from final_enriched_resolved

''')

#df.show(5, truncate=False)

2026-02-25 04:35:12,435 - INFO - 


Registered StarRocks table 'staging_victoria_metrics.node_infiniband_port_data_transmitted_bytes_total' as Spark view 'staging_victoria_metrics__node_infiniband_port_data_transmitted_bytes_total'



2026-02-25 04:35:12,488 - INFO - 


Registered StarRocks table 'staging_victoria_metrics.node_infiniband_port_data_received_bytes_total' as Spark view 'staging_victoria_metrics__node_infiniband_port_data_received_bytes_total'



2026-02-25 04:35:12,535 - INFO - 


Registered StarRocks table 'staging_victoria_metrics.dcgm_fi_prof_pipe_tensor_active' as Spark view 'staging_victoria_metrics__dcgm_fi_prof_pipe_tensor_active'



2026-02-25 04:35:12,582 - INFO - 


Registered StarRocks table 'staging_victoria_metrics.redfish_stats_power_reading' as Spark view 'staging_victoria_metrics__redfish_stats_power_reading'



2026-02-25 04:35:12,627 - INFO - 


Registered StarRocks table 'staging_victoria_metrics.dcgm_fi_dev_power_usage' as Spark view 'staging_victoria_

In [6]:
# ========================================
# SAVE DIRECTLY TO PARQUET (no validation)
# ========================================
# Write the query results directly to parquet without triggering full execution
# This streams data from StarRocks and writes incrementally

import time

print("Writing query results directly to parquet...")
print("This will take time but avoids memory issues...")
print()

output_path = "/tmp/df_union_raw_v2.parquet"

start_time = time.time()

try:
    # Write directly - this streams data and doesn't load everything into memory
    df.write.mode("overwrite").parquet(output_path)
    
    write_time = time.time() - start_time
    print(f"✓ Written to: {output_path}")
    print(f"  Time: {write_time:.1f}s ({write_time/60:.1f} minutes)")
    
    # Now read it back from parquet (this will be much faster to work with)
    print("\nReading back from parquet for validation...")
    df_parquet = spark.read.parquet(output_path)
    df_parquet.cache()
    
    # Now we can safely validate
    row_count = df_parquet.count()
    print(f"✓ Row count: {row_count:,}")
    
    print("\nSample rows:")
    df_parquet.show(5, truncate=False)
    
    print("\n✓ Data saved and validated!")
    print(f"  Use df_parquet for subsequent operations")
    
    # Replace df with the parquet version for future use
    df = df_parquet
    
except Exception as e:
    print(f"✗ Error: {str(e)[:300]}")
    import traceback
    traceback.print_exc()


Writing query results directly to parquet...
This will take time but avoids memory issues...



26/02/25 04:50:26 ERROR Executor: Exception in task 0.0 in stage 0.0 (TID 0)/ 1]
java.sql.SQLSyntaxErrorException: Query reached its timeout of 900 seconds, please increase the 'query_timeout' session variable, pending time:0
	at com.mysql.cj.jdbc.exceptions.SQLError.createSQLException(SQLError.java:112)
	at com.mysql.cj.jdbc.exceptions.SQLExceptionsMapping.translateException(SQLExceptionsMapping.java:114)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeInternal(ClientPreparedStatement.java:988)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeQuery(ClientPreparedStatement.java:1056)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD.compute(JDBCRDD.scala:304)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:367)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:331)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:367)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala

✗ Error: An error occurred while calling o158.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (capacity-finance-analytics-v2-0 executor driver): java.sql.SQLSyntaxErrorException: Query re


26/02/25 04:50:26 ERROR TaskSetManager: Task 0 in stage 0.0 failed 1 times; aborting job
26/02/25 04:50:26 ERROR FileFormatWriter: Aborting job 339b5128-f603-4f5e-82d5-d703b3e4df24.
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (capacity-finance-analytics-v2-0 executor driver): java.sql.SQLSyntaxErrorException: Query reached its timeout of 900 seconds, please increase the 'query_timeout' session variable, pending time:0
	at com.mysql.cj.jdbc.exceptions.SQLError.createSQLException(SQLError.java:112)
	at com.mysql.cj.jdbc.exceptions.SQLExceptionsMapping.translateException(SQLExceptionsMapping.java:114)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeInternal(ClientPreparedStatement.java:988)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeQuery(ClientPreparedStatement.java:1056)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD.compute(JDBCRDD.scala:304)
	a

In [ ]:
print("\n6. Saving complete dataset...")
complete_output_path = "/tmp/df_union_raw_v2.parquet"
df.write.mode("overwrite").parquet(complete_output_path)
print(f"   ✓ Saved to: {complete_output_path}")

In [ ]:
# ========================================
# CREATE TABLE IF NOT EXISTS
# ========================================

# Define CREATE TABLE DDL based on the dataframe schema
CREATE_TABLE_DDL = """
CREATE TABLE IF NOT EXISTS sandbox_finance.capacity_finance_dcgm_metrics_union_raw_v2 (
    datestamp DATETIME NOT NULL,
    node VARCHAR(255),
    gpu_util DOUBLE,
    tensor_util DOUBLE,
    chip_power DOUBLE,
    region VARCHAR(100),
    cluster VARCHAR(255),
    cluster_org VARCHAR(255),
    cw_sku VARCHAR(255),
    model VARCHAR(255),
    serial VARCHAR(255),
    redfish_power DOUBLE,
    customer_name VARCHAR(500),
    flag_is_coreweave TINYINT,
    model_imputed VARCHAR(255),
    product VARCHAR(100),
    product_resolved VARCHAR(100),
    customer_segment VARCHAR(100),
    is_training VARCHAR(50),
    is_multinode VARCHAR(50),
    gpu_count_expected INT,
    peak_tflops_unit INT,
    peak_power_unit INT,
    product_segment VARCHAR(100),
    tensor_tflops DOUBLE,
    gpu_peak_power_node_w INT
)
DUPLICATE KEY(datestamp, node)
PARTITION BY RANGE(datestamp) ()
DISTRIBUTED BY HASH(node) BUCKETS 32
PROPERTIES(
    "replication_num" = "1",
    "storage_medium" = "HDD",
    "enable_persistent_index" = "true"
)
"""

print("Checking if table exists...")
try:
    # Try to query the table
    test_query = star.sql("SELECT 1 FROM sandbox_finance.capacity_finance_dcgm_metrics_union_raw_v2 LIMIT 1")
    print("✓ Table sandbox_finance.capacity_finance_dcgm_metrics_union_raw_v2 already exists")
except Exception as e:
    print(f"Table doesn't exist. Creating table...")
    print(f"Error was: {str(e)}")
    
    # Create the table using direct SQL via engine
    try:
        with engine.connect() as conn:
            conn.execute(text(CREATE_TABLE_DDL))
            conn.commit()
        print("✓ Table created successfully!")
        
        # Verify creation
        verify = star.sql("SELECT 1 FROM sandbox_finance.capacity_finance_dcgm_metrics_union_raw_v2 LIMIT 1")
        print("✓ Table verified - ready for data upload")
    except Exception as create_err:
        print(f"✗ Failed to create table: {str(create_err)}")
        raise


In [ ]:
# ========================================
# UPLOAD TO STARROCKS (FULL DATA)
# ========================================
import time

print("\n" + "="*60)
print("UPLOADING TO STARROCKS")
print("="*60)

# Get credentials from keyring
starrocks_username = keyring.get_password("starrocks", "username")
starrocks_password = keyring.get_password("starrocks", "password")
assert starrocks_username and starrocks_password, "No creds in keyring. Please run Cell 3 first."

# Configuration
STARROCKS_HOST = "kube-starrocks-warehouse-fe-service.starrocks.svc.cluster.local"
STARROCKS_HTTP_PORT = 8030
STARROCKS_JDBC_PORT = 9030
STARROCKS_DB = "sandbox_finance"
STARROCKS_TABLE = "capacity_finance_dcgm_metrics_union_raw_v2"
FULL_TABLE_NAME = f"{STARROCKS_DB}.{STARROCKS_TABLE}"

print(f"\nConnection details:")
print(f"  Host: {STARROCKS_HOST}")
print(f"  Database: {STARROCKS_DB}")
print(f"  Table: {STARROCKS_TABLE}")
print(f"  User: {starrocks_username}")


# Read the COMPLETE imputed data
complete_output_path = "/tmp/df_union_raw_v2.parquet"
print(f"\nReading data from: {complete_output_path}")
sdf_upload = spark.read.parquet(complete_output_path)

row_count = sdf_upload.count()
print(f"Total rows to upload: {row_count:,}")

# Show schema
print(f"\nSchema ({len(sdf_upload.columns)} columns):")
sdf_upload.printSchema()

# Show sample of data to verify
print("\nSample of data to upload:")
sdf_upload.show(5)

# Check current table status
print(f"\nChecking current table status...")
try:
    current_count = star.sql(f"SELECT COUNT(*) as row_count FROM {FULL_TABLE_NAME}")
    print(f"Current row count in table:")
    current_count.show()
    current_rows = current_count.collect()[0]['row_count']
    
    if current_rows > 0:
        print(f"\n⚠️  WARNING: Table contains {current_rows:,} rows that will be DELETED!")
except Exception as e:
    print(f"Could not query table (may not exist yet): {str(e)}")
    current_rows = 0

# Confirmation
print(f"\nThis will:")
print(f"  1. TRUNCATE table {FULL_TABLE_NAME} (delete all {current_rows:,} existing rows)")
print(f"  2. Upload {row_count:,} new rows")
confirm = input("\nType 'yes' to proceed: ")
if confirm.lower() != 'yes':
    print("Upload cancelled by user.")
    raise SystemExit("Upload cancelled")

# Step 1: Truncate table
print(f"\n" + "="*60)
print("STEP 1: TRUNCATING TABLE")
print("="*60)
try:
    print(f"Truncating {FULL_TABLE_NAME}...")
    # TRUNCATE returns None, not a DataFrame
    star.sql(f"TRUNCATE TABLE {FULL_TABLE_NAME}")
    print(f"✓ Table truncated successfully")
    
    # Verify truncation
    verify_empty = star.sql(f"SELECT COUNT(*) as row_count FROM {FULL_TABLE_NAME}")
    print(f"Row count after truncate:")
    verify_empty.show()
except Exception as e:
    print(f"✗ Failed to truncate table: {str(e)}")
    raise

# Step 2: Upload data
print(f"\n" + "="*60)
print("STEP 2: UPLOADING DATA")
print("="*60)
print(f"Starting upload of {row_count:,} rows...")
upload_start = time.time()

try:
    # Use direct Spark writer in APPEND mode (table is already empty from truncate)
    sdf_upload.write \
        .format("starrocks") \
        .option("starrocks.table.identifier", FULL_TABLE_NAME) \
        .option("starrocks.fe.http.url", f"{STARROCKS_HOST}:{STARROCKS_HTTP_PORT}") \
        .option("starrocks.fe.jdbc.url", f"jdbc:mysql://{STARROCKS_HOST}:{STARROCKS_JDBC_PORT}/{STARROCKS_DB}") \
        .option("starrocks.user", starrocks_username) \
        .option("starrocks.password", starrocks_password) \
        .option("starrocks.write.properties.format", "json") \
        .option("starrocks.write.properties.timeout", "3600") \
        .mode("append") \
        .save()
    
    upload_time = time.time() - upload_start
    
    print(f"\n✓ Upload complete!")
    print(f"  Rows uploaded: {row_count:,}")
    print(f"  Upload time: {upload_time:.1f}s ({upload_time/60:.1f} minutes)")
    print(f"  Rate: {row_count/upload_time:,.0f} rows/sec")
    
    # Verify final count
    print(f"\nVerifying final data...")
    final_count = star.sql(f"SELECT COUNT(*) as row_count FROM {FULL_TABLE_NAME}")
    print(f"Final row count in table:")
    final_count.show()
    
    final_rows = final_count.collect()[0]['row_count']
    if final_rows == row_count:
        print(f"\n✓ Row count matches: {final_rows:,} = {row_count:,}")
    else:
        print(f"\n⚠️  WARNING: Row count mismatch!")
        print(f"  Expected: {row_count:,}")
        print(f"  Actual: {final_rows:,}")
        print(f"  Difference: {abs(final_rows - row_count):,}")
    
except Exception as e:
    upload_time = time.time() - upload_start
    print(f"\n✗ Upload failed after {upload_time:.1f}s!")
    print(f"  Error type: {type(e).__name__}")
    print(f"  Error message: {str(e)}")
    print(f"\nFull error details:")
    import traceback
    traceback.print_exc()
    raise

print("\n" + "="*60)
print("✓ UPLOAD COMPLETE")
print("="*60)